# Prefix Length Ablation (Table 3)

Reproduces **Table 3** from the DTR paper.

This notebook evaluates how the length of the observed prefix (number of generated tokens)
affects DTR's predictive power. Since DTR is computed from the JSD settling pattern,
the question is: how many tokens must we observe before DTR becomes a reliable indicator?

Key finding: **prefix=50 tokens** achieves the best cost-accuracy tradeoff, meaning DTR
can make accurate predictions very early in the generation process.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

sns.set_theme(style="whitegrid")

from dtr.analysis.sensitivity import sweep_prefix_lengths
from dtr.analysis.correlation import compute_binned_correlation
from dtr.aggregation.strategies import run_all_strategies, SampleResult
from dtr.aggregation.cost import compute_selective_cost, compute_cost_ratio
from dtr.utils.io import load_json

## Configuration

In [ ]:
MODEL = "deepseek-r1"
BENCHMARK = "aime_2025"  # Table 3 focuses on AIME 2025

METRICS_DIR = Path("..") / "results" / "metrics"
RAW_DIR = Path("..") / "results" / "raw" / MODEL / BENCHMARK

# Prefix lengths to evaluate
PREFIX_LENGTHS = [50, 100, 500, 1000, 2000, -1]  # -1 = full sequence
PREFIX_LABELS = ["50", "100", "500", "1000", "2000", "Full"]

## Load JSD Matrices and Per-Sample Data

In [ ]:
# Load per-sample metrics
metrics_path = METRICS_DIR / MODEL / BENCHMARK / "per_sample_metrics.json"

if metrics_path.exists():
    data = load_json(str(metrics_path))
    print(f"Loaded {len(data)} samples from {metrics_path}")
else:
    print(f"WARNING: {metrics_path} not found")
    data = []

accuracies = np.array([d["correct"] for d in data], dtype=float) if data else np.array([])
print(f"Baseline accuracy: {accuracies.mean():.1%}" if len(accuracies) > 0 else "No data")

## Compute DTR at Different Prefix Lengths

For each prefix length, we truncate the JSD matrix and recompute DTR.
Then we measure the binned correlation with accuracy.

In [ ]:
# Sweep prefix lengths
prefix_results = sweep_prefix_lengths(
    data,
    accuracies,
    prefix_lengths=PREFIX_LENGTHS,
)

# Display results
for pl, label in zip(PREFIX_LENGTHS, PREFIX_LABELS):
    if pl in prefix_results:
        res = prefix_results[pl]
        print(f"Prefix={label:>5s}: correlation={res['correlation']:.3f}, "
              f"accuracy={res.get('accuracy', 'N/A')}, "
              f"cost_ratio={res.get('cost_ratio', 'N/A')}")

## Table 3: Prefix Length Ablation

Shows correlation, Think@n accuracy, and cost savings for each prefix length.

In [ ]:
table_rows = []
for pl, label in zip(PREFIX_LENGTHS, PREFIX_LABELS):
    if pl not in prefix_results:
        continue
    res = prefix_results[pl]
    table_rows.append({
        "Prefix Length": label,
        "Pearson r (DTR vs Acc)": f"{res['correlation']:.3f}",
        "Think@16 Accuracy": f"{res.get('think_at_n_accuracy', '-')}",
        "Cost Ratio": f"{res.get('cost_ratio', '-')}",
        "Cost Savings": f"{res.get('cost_savings', '-')}",
    })

table3_df = pd.DataFrame(table_rows)
table3_df.set_index("Prefix Length", inplace=True)

# Highlight optimal row (prefix=50)
def highlight_optimal(row):
    if row.name == "50":
        return ["background-color: #c6efce; font-weight: bold"] * len(row)
    return [""] * len(row)

styled_table = table3_df.style.apply(highlight_optimal, axis=1)
styled_table.set_caption("Table 3: Prefix Length Ablation (AIME 2025)")
styled_table

## Bar Chart: Correlation by Prefix Length

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f"Prefix Length Ablation: {MODEL} on {BENCHMARK.replace('_', ' ').title()}",
    fontsize=15, fontweight="bold", y=1.02,
)

# Extract data for plotting
plot_labels = []
plot_corrs = []
plot_costs = []

for pl, label in zip(PREFIX_LENGTHS, PREFIX_LABELS):
    if pl in prefix_results:
        plot_labels.append(label)
        plot_corrs.append(prefix_results[pl]["correlation"])
        plot_costs.append(prefix_results[pl].get("cost_ratio", 1.0))

x = np.arange(len(plot_labels))
colors = [sns.color_palette()[3] if l == "50" else sns.color_palette()[0] for l in plot_labels]

# Subplot 1: Correlation
ax1 = axes[0]
bars1 = ax1.bar(x, plot_corrs, color=colors, edgecolor="white", width=0.6)
ax1.set_xticks(x)
ax1.set_xticklabels(plot_labels)
ax1.set_xlabel("Prefix Length (tokens)", fontsize=12)
ax1.set_ylabel("Pearson r (DTR vs Accuracy)", fontsize=12)
ax1.set_title("Correlation Strength", fontsize=13)
ax1.set_ylim(0, 1)

for bar, val in zip(bars1, plot_corrs):
    ax1.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
        f"{val:.3f}", ha="center", fontsize=10, fontweight="bold",
    )

# Subplot 2: Cost ratio
ax2 = axes[1]
bars2 = ax2.bar(x, plot_costs, color=colors, edgecolor="white", width=0.6)
ax2.set_xticks(x)
ax2.set_xticklabels(plot_labels)
ax2.set_xlabel("Prefix Length (tokens)", fontsize=12)
ax2.set_ylabel("Normalized Cost", fontsize=12)
ax2.set_title("Inference Cost", fontsize=13)

for bar, val in zip(bars2, plot_costs):
    ax2.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
        f"{val:.2f}x", ha="center", fontsize=10, fontweight="bold",
    )

plt.tight_layout()
plt.savefig("../results/figures/table3_prefix_ablation.pdf", bbox_inches="tight", dpi=300)
plt.show()

## Accuracy-Cost Tradeoff by Prefix Length

Scatter plot showing each prefix length as a point in accuracy-cost space.
prefix=50 achieves the best tradeoff (high correlation at minimal cost).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for i, (label, corr, cost) in enumerate(zip(plot_labels, plot_corrs, plot_costs)):
    marker_color = sns.color_palette()[3] if label == "50" else sns.color_palette()[0]
    marker_size = 150 if label == "50" else 80
    ax.scatter(
        cost, corr,
        s=marker_size, color=marker_color,
        edgecolor="black", linewidth=1, zorder=5,
    )
    ax.annotate(
        f"prefix={label}",
        (cost, corr),
        textcoords="offset points",
        xytext=(10, 5),
        fontsize=10,
        fontweight="bold" if label == "50" else "normal",
    )

ax.set_xlabel("Normalized Inference Cost", fontsize=13)
ax.set_ylabel("Pearson r (DTR vs Accuracy)", fontsize=13)
ax.set_title("Prefix Length: Accuracy-Cost Tradeoff", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

## Key Takeaway

A prefix of just **50 tokens** is sufficient for DTR to achieve strong correlation with
final accuracy. This means Think@n can make early routing decisions with minimal compute
overhead, dramatically reducing inference cost while maintaining predictive quality.